- function

| 함수  | 변환 전 | 변환 후 |
|-------|--------|--------|
| Input | 문자형 | 숫자형 |
| Put   | 숫자형 | 문자형 |

- `PROC IMPORT`

```sas
proc import                                /* 외부 파일 → SAS로 불러오는 프로시저 */

    datafile="&Path\data.xlsx"   /* 불러올 파일 경로 (매크로 변수 사용) */
                                           /* 예: C:\project\data.xlsx */

    dbms=xlsx                              /* 파일 형식 지정 (Excel .xlsx) */

    replace                                /* 같은 이름의 dataset 있으면 덮어쓰기 */

    out=data1;                            /* 결과를 data1으로 저장 */

    sheet="sheet1";                        /* Excel의 특정 시트 선택 */

run;
```

- `PROC MEANS`

```sas
proc means data=advs n mean std median min max;
    var aval;
    output out=stat
        n=n
        mean=mean
        std=std;
run;
```

- ttest

- one-sample ttest

```sas
proc ttest data=advs h0=0;
    var aval;
run;
```

- two-sample ttest

```sas
proc ttest data=advs;
    class trta;
    var aval;
run;
```

- paired ttest

```sas
proc ttest data=advs;
    paired base*week4;
run;
```

- Wilcoxon

```sas
proc npar1way data=advs wilcoxon;
    class trta;
    var aval;
run;
```

- ANOVA

```sas
proc anova data=advs;
    class trta;
    model aval = trta;
run;
```

```sas
proc glm data=advs;
    class trta;
    model aval = trta;
run;
```

- Kruskal-Wallis

```sas
proc npar1way data=advs;
    class trta;
    var aval;
run;
```

- chi square

```sas
proc freq data=adae;
    tables trta*aeflag / chisq;
run;
```

- Fisher's exact Test

```sas
proc freq data=adae;
    tables trta*aeflag / fisher;
run;
```

- `BE guidance`

**2 by 2**

```sas
proc mixed data=&data ;                         /* &data 데이터셋으로 mixed model 실행 */

    class TRTA(ref="&ref")                      /* 처리군 변수. &ref를 기준군(reference)으로 설정 */
          USUBJID                               /* 대상자 ID */
          APERIOD                               /* period 변수 */
          TRTSEQA;                              /* treatment sequence 변수 */

    model &para = TRTSEQA APERIOD TRTA;         /* 종속변수 &para를 sequence, period, treatment로 설명 */
                                                /* 보통 &para는 log-transformed PK parameter일 수 있음 */

    random USUBJID(TRTSEQA);                    /* sequence 내 subject를 random effect로 지정 */
                                                /* 같은 sequence 안의 대상자 간 변동 고려 */

    estimate "&TRT2header/&TRT1header"          /* treatment contrast 이름 */
             TRTA 1 -1                          /* TRTA 두 수준 간 차이 추정 */
             / cl alpha=0.1;                    /* 신뢰구간 출력, alpha=0.1 → 90% CI */

    lsmeans TRTA;                               /* treatment별 least squares mean 산출 */

run;
```

**2 by 4**

Method A - `PROC GLM` (Recommended)

```sas
proc glm data=replicate;                              /* replicate design dataset 사용 */

    class formulation subject period sequence;        /* 범주형 변수 선언 */
                                                       /* formulation = T/R */
                                                       /* subject = 대상자 */
                                                       /* period = 투여 period */
                                                       /* sequence = 투여 순서 */

    model logDATA = sequence 
                    subject(sequence) 
                    period 
                    formulation;                       /* log PK parameter를 분석 */
                                                       /* sequence, period, formulation 효과 포함 */
                                                       /* subject(sequence)를 fixed effect처럼 포함 */

    estimate "test-ref" formulation -1 1;              /* Test - Reference 차이 추정 */
                                                       /* level 순서에 따라 부호 확인 필요 */

    test h=sequence e=subject(sequence);               /* sequence effect 검정 시 */
                                                       /* subject(sequence)를 error term으로 사용 */

    lsmeans formulation / adjust=t 
                          pdiff=control("R") 
                          CL alpha=0.10;               /* formulation별 LS mean */
                                                       /* Reference("R") 대비 비교 */
                                                       /* alpha=0.10 → 90% CI */

run;
```

**2 by 4**

Method B - `PROC MISED`

```sas
proc mixed data=replicate;                            /* mixed model 사용 */

    class formulation subject period sequence;         /* 범주형 변수 선언 */

    model logDATA = sequence 
                    period 
                    formulation;                       /* fixed effect */
                                                       /* sequence, period, formulation 포함 */

    random subject(sequence);                          /* subject(sequence)를 random effect로 지정 */
                                                       /* sequence 내 subject variability 반영 */

    estimate "test-ref" formulation -1 1 
             / CL alpha=0.10;                          /* Test - Reference 차이 */
                                                       /* 90% CI 출력 */

run;
```

**2 by 4**

Method C - `PROC MIXED` (FDA style)

```sas
proc mixed data=replicate;                            /* FDA-style replicate design mixed model */

    classes sequence subject period formulation;       /* 범주형 변수 선언 */

    model logDATA = sequence 
                    period 
                    formulation 
                    / ddfm=satterth;                   /* fixed effect */
                                                       /* ddfm=satterth = Satterthwaite 자유도 보정 */

    random formulation / type=FA0(2) 
                         sub=subject 
                         G;                            /* subject별 formulation random effect */
                                                       /* subject × formulation interaction 반영 */
                                                       /* G = covariance matrix 출력 */

    repeated / grp=formulation 
               sub=subject;                            /* formulation별 residual variance 허용 */
                                                       /* Test와 Reference의 within-subject variance 분리 */

    estimate 'test-ref' formulation -1 1 
             / CL alpha=0.10;                          /* Test - Reference 차이 */
                                                       /* 90% CI 출력 */

run;
```